<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/headlineGenerationNLP_t5_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📰 T5-Small News Headline Generation Model

This notebook implements a complete NLP pipeline for **automatic news headline generation** using the :contentReference[oaicite:0]{index=0}, a lightweight encoder–decoder transformer designed for sequence-to-sequence learning tasks.

---

## Model Architecture

The system is based on the :contentReference[oaicite:1]{index=1} architecture, which treats every NLP task as a **text-to-text problem**.

- **Encoder:** Converts input news article into contextual embeddings  
- **Decoder:** Generates the corresponding headline token-by-token  
- **Framework:** Hugging Face Transformers (`T5ForConditionalGeneration`)  
- **Task Type:** Abstractive summarization / headline generation  

This makes the model suitable for generating concise, human-like headlines from long articles.

---

## Training Strategy

The model is fine-tuned using supervised learning on paired data:

- **Input:** News article text  
- **Target:** Reference headline  

Training is performed using the `Seq2SeqTrainer`, which automates:
- Forward pass through the transformer
- Loss computation using cross-entropy
- Backpropagation and optimization

### Key Hyperparameters
- Learning Rate: `3e-4`
- Batch Size: `4`
- Epochs: `2`
- Weight Decay: `0.01`
- Precision: FP16 (if GPU available)
- Evaluation Strategy: Per epoch

---

## Evaluation Method

Model performance is measured using **ROUGE metrics** via the `evaluate` library:

- **ROUGE-1:** Word overlap accuracy  
- **ROUGE-2:** Phrase-level similarity  
- **ROUGE-L:** Longest common subsequence matching  

These metrics evaluate how close generated headlines are to human-written references.

---

## Inference Behavior

During prediction, the model:
1. Receives a news article
2. Encodes it using the :contentReference[oaicite:2]{index=2} encoder
3. Generates output using autoregressive decoding
4. Produces a single concise headline

---


In [ ]:
!pip install -q transformers==4.41.2 datasets sentencepiece accelerate evaluate rouge_score pandas scikit-learn

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

DATA_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"

train_df = pd.read_csv(os.path.join(DATA_DIR, "t5_small_train.csv"))[["model_input","model_target"]].dropna()
val_df   = pd.read_csv(os.path.join(DATA_DIR, "t5_small_validation.csv"))[["model_input","model_target"]].dropna()
test_df  = pd.read_csv(os.path.join(DATA_DIR, "t5_small_test.csv"))[["model_input","model_target"]].dropna()

print(len(train_df), len(val_df), len(test_df))

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

max_input = 512
max_output = 64

def preprocess(batch):
    inputs = tokenizer(
        batch["model_input"],
        max_length=max_input,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        batch["model_target"],
        max_length=max_output,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset   = val_dataset.map(preprocess, batched=True)
test_dataset  = test_dataset.map(preprocess, batched=True)

train_dataset = train_dataset.remove_columns(["model_input","model_target"])
val_dataset   = val_dataset.remove_columns(["model_input","model_target"])
test_dataset  = test_dataset.remove_columns(["model_input","model_target"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/t5_small_results",
    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=2,
    weight_decay=0.01,

    predict_with_generate=True,
    fp16=torch.cuda.is_available(),

    logging_steps=50,
    save_total_limit=2
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

print(trainer.evaluate(test_dataset))



## Summary

This pipeline demonstrates how a pre-trained transformer like :contentReference[oaicite:3]{index=3} can be effectively fine-tuned for real-world NLP tasks such as news headline generation, achieving strong performance with minimal architecture changes and efficient training using Hugging Face tools.

In [ ]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SAVE_PATH = "/content/drive/MyDrive/t5_small_model"

model.save_pretrained(SAVE_PATH, safe_serialization=False)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved successfully!")

Model saved successfully!


#FINAL TEST RESULTS EVALUATION

In [ ]:
import torch
import numpy as np
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

predictions = []
references = []

batch_size = 8

with torch.no_grad():
    for i in tqdm(range(0, len(test_dataset), batch_size)):

        batch = test_dataset[i:i+batch_size]

        input_ids = torch.tensor(batch["input_ids"]).to(device)
        attention_mask = torch.tensor(batch["attention_mask"]).to(device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=64,
            num_beams=4
        )

        preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        labels = batch["labels"]
        labels = np.where(np.array(labels) != -100, labels, tokenizer.pad_token_id)
        refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

        predictions.extend(preds)
        references.extend(refs)

results = rouge.compute(
    predictions=predictions,
    references=references
)

print("\n🔥 FINAL TEST RESULTS:")
for k, v in results.items():
    print(f"{k}: {round(v, 4)}")

  0%|          | 0/595 [00:00<?, ?it/s]/tmp/ipykernel_1643/4248185744.py:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(batch["input_ids"]).to(device)
/tmp/ipykernel_1643/4248185744.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(batch["attention_mask"]).to(device)
100%|██████████| 595/595 [26:25<00:00,  2.67s/it]



🔥 FINAL TEST RESULTS:
rouge1: 0.3705
rouge2: 0.1741
rougeL: 0.3409
rougeLsum: 0.341


#ROUGE threshold based

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
import evaluate

rouge = evaluate.load("rouge")

ROUGE_THRESHOLD = 0.2

y_true = []
y_pred = []

for i in range(len(predictions)):

    ref_list = [references[i]]
    pred_list = [predictions[i]]

    scores = rouge.compute(predictions=pred_list, references=ref_list)

    if scores['rougeLsum'] >= ROUGE_THRESHOLD:
        y_pred.append(1)
    else:
        y_pred.append(0)

    y_true.append(1)
accuracy = accuracy_score(y_true, y_pred)

print("🔥 Accuracy (ROUGE threshold based):", round(accuracy, 4))

🔥 Accuracy (ROUGE threshold based): 0.7252


# T5 FULL EVALUATION
# ACCURACY + CONFUSION MATRIX + ROC CURVE

# 14. INTERACTIVE TESTING




In [ ]:


def generate_headline(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding="max_length"
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )

    headline = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return headline



print("\n🔥 Headline Generator Ready! Type 'exit' to stop.\n")

while True:
    user_text = input("Enter news text: ")

    if user_text.lower() == "exit":
        print("Stopped.")
        break

    result = generate_headline(user_text)
    print("\n📰 Generated Headline:", result)
    print("-" * 60)


🔥 Headline Generator Ready! Type 'exit' to stop.

Enter news text: The Egyptian government announced new economic reforms aimed at boosting exports and reducing inflation.

📰 Generated Headline: Egypt announces new economic reforms aimed at exports
------------------------------------------------------------
Enter news text: The government approved a new budget plan to improve healthcare services across the country.

📰 Generated Headline: Government approves new budget plan to improve healthcare services across the country
------------------------------------------------------------
Enter news text: Scientists at Cairo University announced a breakthrough in renewable energy storage that could significantly reduce electricity costs in Egypt over the next decade.

📰 Generated Headline: Scientists announce breakthrough in renewable energy storage
------------------------------------------------------------
Enter news text: The Egyptian parliament discussed a new law aimed at regulating s